# Sampling information with the same open estimator

This working comparator freezes the same invented inputs, development-only evidence, schedule design and engineering settings as the native Task 2 CLI. Each schedule removes raw rows before the shared parser, initialization, exact estimator and keyed uncertainty. No omitted value or full-curve state is supplied to a reduced calculation.

The notebook calls the shared calculation and report functions directly; it does not invoke a CLI or provide another estimator. All scheduled failures and denominators remain visible. Total secretion includes basal over measured 0–120 minutes in pmol, after minus before; C-peptide AUC and glucose are separate context. Full curves are reference analyses, not biological truth.

An experienced human analyst must still qualify this comparator before timed sessions. The technical Python cell runner is not a Jupyter UI or human usability assessment. Choose fresh output paths. Export a canonical example and import its development-only projection before using the default paths.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "experiments/mmtt_sampling").is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PARAMETERS = globals().get("PARAMETERS", {})
INPUT = ROOT / "results/ot056-sampling-notebook-input"
PACKAGE_FILE = Path(PARAMETERS.get("PACKAGE_FILE", INPUT / "package.json"))
CSV_FILE = Path(PARAMETERS.get("CSV_FILE", INPUT / "observations.csv"))
SETTINGS_FILE = Path(PARAMETERS.get("SETTINGS_FILE", INPUT / "settings.json"))
DESIGN_FILE = Path(PARAMETERS.get("DESIGN_FILE", INPUT / "sampling-design.json"))
DEVELOPMENT_IMPORT = Path(
    PARAMETERS.get("DEVELOPMENT_IMPORT", ROOT / "results/ot056-sampling-development-import")
)
OUTPUT = Path(PARAMETERS.get("OUTPUT_DIRECTORY", ROOT / "results/ot056-sampling-notebook-output"))

from experiments.catalogue.storage import dump, fresh  # noqa: E402 -- root path initialized above

if "OUTPUT_DIRECTORY" not in PARAMETERS:
    OUTPUT = fresh(OUTPUT)

## Freeze selection evidence and original inputs

Selection participants remain disjoint from evaluation participants. The freezer checks that development metadata and raw cells match the original study, binds the source/runtime, and performs no evaluation fitting. These allocations exercise synthetic software isolation; they are not a blinded empirical experiment.

In [ ]:
from experiments.mmtt_sampling.plan import freeze, load_plan

freeze(PACKAGE_FILE, CSV_FILE, SETTINGS_FILE, DESIGN_FILE, DEVELOPMENT_IMPORT, OUTPUT / "frozen")
plan, package, raw = load_plan(OUTPUT / "frozen", check_runtime=True)
assert plan["kind"] == "synthetic" and plan["human_execution_authorized"] is False
print("Participants:", len(package["participants"]))
print("Selection participants:", plan["design"]["selection_participants"])
print("Fixed window:", plan["design"]["full_times_minutes"])

## Recalculate each full and reduced schedule

Shared raw projection precedes analyte parsing. Each independent native plan adds its own required-time gate; a missing requested measurement invalidates its visit before the estimator. All other settings remain identical. The shared `evaluate` function retains every variant, paired perturbation and failure. Masking refusals remain planned, unattempted schedules rather than disappearing from the report.

In [ ]:
from copy import deepcopy

from experiments.mmtt_sampling.run import initial_records, project, schedule_map
from experiments.observation_analysis.analysis import evaluate, report_from_attempts
from experiments.observation_analysis.plan import freeze as freeze_native
from experiments.observation_analysis.plan import load_plan as load_native
from experiments.observation_analysis.report import render as render_native

records = initial_records(plan)
for name, times in schedule_map(plan).items():
    record = records[name]
    try:
        masked, inventory = project(raw, package, plan, times)
    except (ValueError, OSError) as error:
        record.update(status="mask_failed", error=str(error), error_type=type(error).__name__)
        continue
    record["inventory"] = inventory
    source = OUTPUT / "inputs" / name
    source.mkdir(parents=True)
    (source / "package.json").write_bytes(raw["package.json"])
    (source / "observations.csv").write_bytes(masked)
    settings = {**deepcopy(plan["settings"]), "required_times_min": list(times)}
    dump(source / "settings.json", settings)
    try:
        freeze_native(
            source / "package.json",
            source / "observations.csv",
            source / "settings.json",
            OUTPUT / "plans" / name,
        )
    except (ValueError, OSError, TypeError, KeyError, OverflowError) as error:
        record.update(status="plan_failed", error=str(error), error_type=type(error).__name__)
        continue
    native_plan, native_package, native_raw = load_native(
        OUTPUT / "plans" / name, check_runtime=True
    )
    attempts = evaluate(native_plan["settings"], native_package, native_raw["observations.csv"])
    native_report = report_from_attempts(native_plan, attempts)
    native_report["execution_status"] = "completed"
    child = OUTPUT / "studies" / name
    child.mkdir(parents=True)
    dump(child / "attempts.json", attempts)
    dump(child / "report.json", native_report)
    (child / "report.md").write_bytes(render_native(native_report))
    record.update(status="completed", report=native_report)
    print(name, "complete paired variants:", native_report["coverage"]["completed_variant_pairs"])

## Compare without hiding failed references

Use the same failure-aware comparison and Markdown renderer as the CLI. The full reference is not true secretion. Unavailable differences remain unavailable; two unresolved directions do not establish preserved information. No empirical equivalence margins, prospective blood-draw approval or product value are inferred.

The notebook report has no native process journal. Technical comparison removes only the CLI report's top-level `execution` journal; all scientific JSON content, complete child reports/attempts and Markdown must otherwise match exactly.

In [ ]:
from experiments.mmtt_sampling.report import build, render

report = build(plan, package, records, "completed")
dump(OUTPUT / "report.json", report)
(OUTPUT / "report.md").write_bytes(render(report))
print("Planned visit points:", report["coverage"]["planned_visit_points"])
print("Planned paired perturbations:", report["coverage"]["planned_draw_pairs"])
try:
    from IPython.display import Markdown, display
except ImportError:
    print((OUTPUT / "report.md").read_text())
else:
    display(Markdown((OUTPUT / "report.md").read_text()))